In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A01T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A04E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A08T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A03E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A09E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A06E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A06T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A02T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A05E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A03T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A07E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A09T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A08E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A07T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A05T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A02E.gdf
/kaggle/input/datasets/moonimint/bci-iv-

In [2]:
# Cell 1: Environment setup, downloading labels, and checking paths
!pip install braindecode -q

# Download true labels directly to working directory
!wget -q https://www.bbci.de/competition/iv/results/ds2a/true_labels.zip -O /kaggle/working/true_labels.zip
!unzip -q -o /kaggle/working/true_labels.zip -d /kaggle/working/true_labels

import os
import torch
import mne
import numpy as np
from scipy.io import loadmat

mne.set_log_level('WARNING')

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

print("\n--- Available Kaggle Inputs ---")
for dirname, _, filenames in os.walk('/kaggle/input'):
    if filenames:
        print(f"{dirname} -> {len(filenames)} files")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.9/626.9 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 11.7 MB/s eta 0:00:00
CUDA available: True
GPU name: Tesla T4

--- Available Kaggle Inputs ---
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf -> 18 files


In [3]:
# Cell 2: Define Exact Epoching Functions for Raw GDF Files

dataset_path = '/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf'
true_labels_path = '/kaggle/working/true_labels'

def get_raw_epochs_T(gdf_path, verbose=False):
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, eog=['EOG-left', 'EOG-central', 'EOG-right'], verbose=verbose)
    events, event_id = mne.events_from_annotations(raw, verbose=verbose)
    
    target_event_id = {
        '769': event_id['769'], '770': event_id['770'], 
        '771': event_id['771'], '772': event_id['772']
    }
    
    epochs = mne.Epochs(raw, events, event_id=target_event_id, 
                        tmin=0, tmax=4.0, baseline=None, preload=True, verbose=verbose)
    epochs_eeg = epochs.copy().pick(picks='eeg') 
    
    drop_indices = []
    if '1023' in event_id:
        rejected_starts = events[events[:, 2] == event_id['1023']][:, 0]
        offset_samples = int(2.0 * raw.info['sfreq'])
        rejected_cue_samples = rejected_starts + offset_samples
        
        epoch_sample_times = epochs.events[:, 0]
        drop_mask = np.isin(epoch_sample_times, rejected_cue_samples)
        drop_indices = np.where(drop_mask)[0]
        epochs_eeg = epochs_eeg.drop(drop_indices, verbose=verbose)
        
    return epochs_eeg, target_event_id

def get_raw_epochs_E(gdf_path, mat_path, verbose=False):
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, eog=['EOG-left', 'EOG-central', 'EOG-right'], verbose=verbose)
    events, event_id = mne.events_from_annotations(raw, verbose=verbose)
    
    target_event_id = {'783': event_id['783']}
    
    epochs = mne.Epochs(raw, events, event_id=target_event_id, 
                        tmin=0, tmax=4.0, baseline=None, preload=True, verbose=verbose)
    epochs_eeg = epochs.copy().pick(picks='eeg')
    
    mat_data = loadmat(mat_path)
    labels = mat_data['classlabel'].flatten() 
    
    drop_indices = []
    if '1023' in event_id:
        rejected_starts = events[events[:, 2] == event_id['1023']][:, 0]
        offset_samples = int(2.0 * raw.info['sfreq'])
        rejected_cue_samples = rejected_starts + offset_samples
        
        epoch_sample_times = epochs.events[:, 0]
        drop_mask = np.isin(epoch_sample_times, rejected_cue_samples)
        drop_indices = np.where(drop_mask)[0]
        epochs_eeg = epochs_eeg.drop(drop_indices, verbose=verbose)
        labels = np.delete(labels, drop_indices) 
        
    assert len(epochs_eeg) == len(labels), "Mismatch between epochs and labels count!"
    return epochs_eeg, labels

print(f"Dataset path set to: {dataset_path}")
print("Epoching functions successfully defined!")

Dataset path set to: /kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf
Epoching functions successfully defined!


In [4]:
# Cell 3: Extract Raw Epochs and Labels for All Subjects

subjects = [f"{i:02d}" for i in range(1, 10)]

all_epochs = {}
all_labels = {}

print("Starting raw epoch extraction... This might take a minute.\n")

for subj in subjects:
    # --- T Session ---
    t_fpath = os.path.join(dataset_path, f"A{subj}T.gdf")
    if os.path.exists(t_fpath):
        ep, t_event_id = get_raw_epochs_T(t_fpath)
        # Map event codes (from GDF) to standard 1-4 class labels
        code_to_class = {t_event_id['769']: 1, t_event_id['770']: 2,
                         t_event_id['771']: 3, t_event_id['772']: 4}
        t_labels = np.array([code_to_class[e[2]] for e in ep.events])
        all_epochs[f"A{subj}T"] = ep
        all_labels[f"A{subj}T"] = t_labels
    else:
        print(f"Missing T session for A{subj}")
    
    # --- E Session ---
    e_fpath = os.path.join(dataset_path, f"A{subj}E.gdf")
    e_mat = os.path.join(true_labels_path, f"A{subj}E.mat")
    if os.path.exists(e_fpath) and os.path.exists(e_mat):
        ep, labels = get_raw_epochs_E(e_fpath, e_mat)
        all_epochs[f"A{subj}E"] = ep
        all_labels[f"A{subj}E"] = labels
    else:
        print(f"Missing E session or mat file for A{subj}")

print(f"\nDone! Successfully epoched: {len(all_epochs)}/18 files")

Starting raw epoch extraction... This might take a minute.



/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying runni


Done! Successfully epoched: 18/18 files


In [5]:
# Cell 4: Dataset Wrapper and Training Function
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import Dataset, DataLoader

class EEGDataset(Dataset):
    def __init__(self, X, y):
        # Convert to float32 for data, long/int64 for labels
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) 

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def train_model(model, train_loader, val_loader, n_epochs=200, lr=0.001, patience=25, weight_decay=0.01, verbose=False, apply_mixup=False, mixup_alpha=0.2):
    model = model.to(device)
    
    # 1. Label Smoothing added here!
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # 2. Cosine Annealing Scheduler added here!
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        # --- Training Phase ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()

            # 3. Mixup Augmentation Logic
            if apply_mixup:
                lam = np.random.beta(mixup_alpha, mixup_alpha)
                index = torch.randperm(X_batch.size(0)).to(device)
                
                mixed_X = lam * X_batch + (1 - lam) * X_batch[index]
                outputs = model(mixed_X)
                
                # Mixed Loss
                loss = lam * criterion(outputs, y_batch) + (1 - lam) * criterion(outputs, y_batch[index])
                
                # Estimate accuracy on the dominant class in the mix
                preds = outputs.argmax(1)
                train_correct += (lam * (preds == y_batch).float() + (1 - lam) * (preds == y_batch[index]).float()).sum().item()
            else:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                train_correct += (outputs.argmax(1) == y_batch).sum().item()

            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)
            train_total += y_batch.size(0)
        
        # Step the learning rate scheduler
        scheduler.step()
        
        train_loss /= train_total
        train_acc = train_correct / train_total

        # --- Validation Phase ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Early Stopping Check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy_module.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_model_state)
    return model, history

print("PyTorch Dataset and Training loop ready!")

PyTorch Dataset and Training loop ready!


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy as copy_module
from torch.utils.data import Dataset, DataLoader

# Global device definition
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Pipeline running on: {device}")

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def train_model(model, train_loader, val_loader, n_epochs=200, lr=0.001, patience=25, weight_decay=0.01, verbose=False, apply_mixup=False, mixup_alpha=0.2):
    model = model.to(device) # device yahan se pick hoga
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    best_val_loss, best_model_state, patience_counter = float('inf'), None, 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            if apply_mixup:
                lam = np.random.beta(mixup_alpha, mixup_alpha)
                index = torch.randperm(X_batch.size(0)).to(device)
                mixed_X = lam * X_batch + (1 - lam) * X_batch[index]
                outputs = model(mixed_X)
                loss = lam * criterion(outputs, y_batch) + (1 - lam) * criterion(outputs, y_batch[index])
                preds = outputs.argmax(1)
                train_correct += (lam * (preds == y_batch).float() + (1 - lam) * (preds == y_batch[index]).float()).sum().item()
            else:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                train_correct += (outputs.argmax(1) == y_batch).sum().item()

            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_total += y_batch.size(0)
        
        scheduler.step()
        train_loss, train_acc = train_loss / train_total, train_correct / train_total
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_loss += criterion(outputs, y_batch).item() * X_batch.size(0)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss, val_acc = val_loss / val_total, val_correct / val_total
        history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss, best_model_state, patience_counter = val_loss, copy_module.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break
    model.load_state_dict(best_model_state)
    return model, history

Pipeline running on: cuda


In [15]:
# Cell 6: Euclidean Alignment, Crop and Helper Functions
def euclidean_alignment(X):
    """Aligns EEG trials using Mean Covariance Matrix."""
    n_trials, n_channels, n_times = X.shape
    covs = np.array([np.cov(trial) for trial in X])
    mean_cov = np.mean(covs, axis=0)
    evals, evecs = np.linalg.eigh(mean_cov)
    evals = np.maximum(evals, 1e-10)
    inv_sqrt_mean_cov = evecs @ np.diag(evals**(-0.5)) @ evecs.T
    return np.array([inv_sqrt_mean_cov @ trial for trial in X])

def crop_array(X, y, crop_len=500, stride=50):
    cropped_X, cropped_y, trial_ids = [], [], []
    for i in range(len(X)):
        start = 0
        while start + crop_len <= X.shape[2]:
            cropped_X.append(X[i][:, start:start+crop_len])
            cropped_y.append(y[i])
            trial_ids.append(i)
            start += stride
    return np.array(cropped_X), np.array(cropped_y), np.array(trial_ids)

print("Alignment and Crop functions ready.")

Alignment and Crop functions ready.


In [16]:
from braindecode.models import Deep4Net

crop_len, stride = 500, 50
all_pretrain_X, all_pretrain_y = [], []

print("Running Pretraining...")
for subj in subjects:
    X, y = all_epochs[f"A{subj}T"].get_data(), all_labels[f"A{subj}T"] - 1
    X_aligned = euclidean_alignment(X) # EA ensure karo define ho
    X_c, y_c, _ = crop_array(X_aligned, y, crop_len, stride)
    all_pretrain_X.append(X_c); all_pretrain_y.append(y_c)

X_pretrain = np.concatenate(all_pretrain_X, axis=0)
y_pretrain = np.concatenate(all_pretrain_y, axis=0)

# Split and Normalize
n_total = len(X_pretrain)
idx = np.random.RandomState(42).permutation(n_total)
X_tr, X_val = X_pretrain[idx[n_total//10:]], X_pretrain[idx[:n_total//10]]
y_tr, y_val = y_pretrain[idx[n_total//10:]], y_pretrain[idx[:n_total//10]]

mean, std = X_tr.mean(axis=(0, 2), keepdims=True), X_tr.std(axis=(0, 2), keepdims=True)
loader_tr = DataLoader(EEGDataset((X_tr - mean)/(std + 1e-8), y_tr), batch_size=128, shuffle=True)
loader_val = DataLoader(EEGDataset((X_val - mean)/(std + 1e-8), y_val), batch_size=128, shuffle=False)

model = Deep4Net(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=0.5)
pretrained_model, hist = train_model(model, loader_tr, loader_val, n_epochs=100, apply_mixup=True)
pretrained_state = pretrained_model.state_dict()
print("Pretraining Complete!")

Running Pretraining...
Pretraining Complete!


In [17]:
# Cell 8: Subject-specific Fine-Tuning & Final Evaluation
def finetune_subject(subject_id, pretrained_state_dict, crop_len=500, stride=50, n_epochs=50, lr=0.0001, patience=15, batch_size=32):
    # Load raw data
    X_train_full = all_epochs[f"A{subject_id}T"].get_data()
    y_train_full = all_labels[f"A{subject_id}T"] - 1
    X_test_full = all_epochs[f"A{subject_id}E"].get_data()
    y_test_full = all_labels[f"A{subject_id}E"] - 1

    # Split
    indices = np.random.RandomState(42).permutation(len(X_train_full))
    n_val = int(0.2 * len(X_train_full))
    
    # Align and Crop Training/Validation
    X_tr_aligned = euclidean_alignment(X_train_full[indices[n_val:]])
    X_val_aligned = euclidean_alignment(X_train_full[indices[:n_val]])
    
    X_tr_c, y_tr_c, _ = crop_array(X_tr_aligned, y_train_full[indices[n_val:]], crop_len, stride)
    X_val_c, y_val_c, _ = crop_array(X_val_aligned, y_train_full[indices[:n_val]], crop_len, stride)

    # Normalize using Global Pretraining Stats
    X_tr_norm = (X_tr_c - pretrain_mean) / (pretrain_std + 1e-8)
    X_val_norm = (X_val_c - pretrain_mean) / (pretrain_std + 1e-8)
    
    train_loader = DataLoader(EEGDataset(X_tr_norm, y_tr_c), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(EEGDataset(X_val_norm, y_val_c), batch_size=batch_size, shuffle=False)

    # Initialize Deep4Net & Load Pretrained Weights
    model = Deep4Net(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=0.5)
    model.load_state_dict(copy_module.deepcopy(pretrained_state_dict))

    # Fine-tune
    finetuned, hist = train_model(model, train_loader, val_loader, n_epochs=n_epochs, lr=lr, patience=patience, apply_mixup=True)

    # Evaluate on E-session (Aligned + Majority Vote)
    X_test_aligned = euclidean_alignment(X_test_full)
    X_test_c, y_test_c, test_trial_ids = crop_array(X_test_aligned, y_test_full, crop_len, stride)
    X_test_norm = (X_test_c - pretrain_mean) / (pretrain_std + 1e-8)
    
    test_loader = DataLoader(EEGDataset(X_test_norm, y_test_c), batch_size=batch_size, shuffle=False)

    finetuned.eval()
    crop_preds = []
    with torch.no_grad():
        for Xb, _ in test_loader:
            crop_preds.extend(finetuned(Xb.to(device)).argmax(1).cpu().numpy())
    
    final_preds = [np.bincount(np.array(crop_preds)[test_trial_ids == i]).argmax() for i in range(len(X_test_full))]
    
    acc = accuracy_score(y_test_full, final_preds)
    kappa = cohen_kappa_score(y_test_full, final_preds)
    
    print(f"Subject A{subject_id}: Test Acc={acc:.4f} | Kappa={kappa:.4f}")
    return {'subject': f"A{subject_id}", 'test_acc': acc, 'test_kappa': kappa}

# Execution
print("Running Fine-Tuning (Raw Data + Deep4Net + EA)...\n")
results = pd.DataFrame([finetune_subject(subj, pretrained_state) for subj in subjects])

print("\n=== Final Summary [RAW DATA + DEEP4NET + EA] ===")
print(results)
print(f"\nMean test accuracy: {results['test_acc'].mean():.4f}")

Running Fine-Tuning (Raw Data + Deep4Net + EA)...

Subject A01: Test Acc=0.8719 | Kappa=0.8292
Subject A02: Test Acc=0.5053 | Kappa=0.3417
Subject A03: Test Acc=0.8388 | Kappa=0.7848
Subject A04: Test Acc=0.5439 | Kappa=0.3940
Subject A05: Test Acc=0.4964 | Kappa=0.3266
Subject A06: Test Acc=0.5767 | Kappa=0.4358
Subject A07: Test Acc=0.8592 | Kappa=0.8125
Subject A08: Test Acc=0.7749 | Kappa=0.7004
Subject A09: Test Acc=0.7576 | Kappa=0.6772

=== Final Summary [RAW DATA + DEEP4NET + EA] ===
  subject  test_acc  test_kappa
0     A01  0.871886    0.829220
1     A02  0.505300    0.341696
2     A03  0.838828    0.784828
3     A04  0.543860    0.394020
4     A05  0.496377    0.326593
5     A06  0.576744    0.435809
6     A07  0.859206    0.812536
7     A08  0.774908    0.700352
8     A09  0.757576    0.677237

Mean test accuracy: 0.6916
